# Day 1 — Getting to know the Indian Railways data
**YTS+ DSEB 2026 · Project B**

---

Indian Railways publishes the complete schedule of every train — which stations it stops at, in order. Someone collected all 4,888 train schedules: 417,080 individual station stops.

From that timetable data, three datasets were built: stations, track connections between stations, and train statistics. Today you will load all three, understand what each column means, and work through exercises to get comfortable with the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'

print('Ready.')

---
## Part 1 — Stations

Load the stations file. Each row is one railway station.

In [ ]:
stations = pd.read_csv(DATA + 'stations_clean.csv')

print(f'Shape: {stations.shape}')
print(f'Columns: {stations.columns.tolist()}')
print()
stations.head(10)

**What each column means:**
- `code` — the official Indian Railways station code (e.g. NDLS = New Delhi, BCT = Mumbai Central)
- `name` — station name
- `state` — Indian state
- `lat`, `lon` — GPS coordinates

In [ ]:
print('Summary statistics:')
print(stations[['lat', 'lon']].describe().round(2))
print()
print('Missing values:')
print(stations.isnull().sum())

In [ ]:
by_state = stations[stations['state'] != ''].groupby('state').size().sort_values(ascending=False)

print('Stations per state (top 20):')
print(by_state.head(20).to_string())

**Exercise 1** — How many stations are in Bihar? Filter to Bihar only and print the shape.

In [ ]:
# Your code here


**Exercise 2** — Find the northernmost station (highest `lat`) and the southernmost station (lowest `lat`).

In [ ]:
# Your code here


In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(stations['lon'], stations['lat'], s=0.4, color='steelblue', alpha=0.5)
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'India — all {len(stations):,} railway stations')
plt.tight_layout()
plt.savefig('images/day1_stations.png', dpi=150)
plt.show()

---
## Part 2 — Track connections

Load the track file. Each row is a physical track connection — two stations directly linked by rail with no station in between.

In [ ]:
track = pd.read_csv(DATA + 'track_edges_enriched.csv')

print(f'Shape: {track.shape}')
print(f'Columns: {track.columns.tolist()}')
print()
track.head(10)

**What each column means:**
- `station_a`, `station_b` — the station codes at each end of the connection
- `name_a`, `name_b` — station names
- `state_a`, `state_b` — states
- `num_trains` — how many different train services run on this connection
- `distance_km` — straight-line distance between the two stations

In [ ]:
print('Track connection statistics:')
print(track[['num_trains', 'distance_km']].describe().round(1))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(track['distance_km'], bins=60, color='steelblue', edgecolor='white')
ax1.set_xlabel('Distance (km)')
ax1.set_ylabel('Number of connections')
ax1.set_title('Gap between neighboring stations')

ax2.hist(track['num_trains'].clip(upper=150), bins=50, color='tomato', edgecolor='white')
ax2.set_xlabel('Number of trains (clipped at 150)')
ax2.set_ylabel('Number of connections')
ax2.set_title('Trains per connection')

plt.tight_layout()
plt.savefig('images/day1_track_dist.png', dpi=150)
plt.show()

In [ ]:
print('10 longest track connections:')
print(track.sort_values('distance_km', ascending=False)
      [['name_a', 'name_b', 'state_a', 'distance_km', 'num_trains']]
      .head(10).to_string(index=False))

In [ ]:
print('10 busiest connections (most trains):')
print(track.sort_values('num_trains', ascending=False)
      [['name_a', 'name_b', 'state_a', 'num_trains', 'distance_km']]
      .head(10).to_string(index=False))

**Exercise 3** — How many connections are entirely within Uttar Pradesh (both `state_a` and `state_b` equal `'Uttar Pradesh'`)?

In [ ]:
# Your code here


**Exercise 4** — How many stations are dead ends — stations that appear in only one connection?

Hint: stack `station_a` and `station_b` into one series with `pd.concat`, use `.value_counts()`, then count how many have a value of 1.

In [ ]:
# Your code here


In [ ]:
# Draw every track connection as a line between its two endpoints
coords = stations.set_index('code')[['lat', 'lon']]

fig, ax = plt.subplots(figsize=(8, 10))
for _, row in track.iterrows():
    a, b = row['station_a'], row['station_b']
    if a in coords.index and b in coords.index:
        ax.plot([coords.loc[a,'lon'], coords.loc[b,'lon']],
                [coords.loc[a,'lat'], coords.loc[b,'lat']],
                color='steelblue', linewidth=0.3, alpha=0.4)

ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'India — {len(track):,} physical track connections')
plt.tight_layout()
plt.savefig('images/day1_track_map.png', dpi=150)
plt.show()

---
## Part 3 — Trains

Load the train statistics file. Each row is one train service.

In [ ]:
trains = pd.read_csv(DATA + 'train_stats.csv')

print(f'Shape: {trains.shape}')
print(f'Columns: {trains.columns.tolist()}')
print()
trains.head(10)

**What each column means:**
- `train_number` — the official train number
- `n_stops` — how many stations this train halts at
- `total_distance_km` — total route length in km
- `avg_km_per_stop` — average distance between each consecutive halt
- `n_skip_segments` — how many times the train passes through a station without stopping

In [ ]:
print(trains[['n_stops', 'total_distance_km', 'avg_km_per_stop']].describe().round(1))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(trains['n_stops'], bins=50, color='steelblue', edgecolor='white')
ax1.set_xlabel('Number of stops')
ax1.set_ylabel('Number of trains')
ax1.set_title('How many stations does each train stop at?')

ax2.hist(trains['avg_km_per_stop'].clip(upper=40), bins=50, color='tomato', edgecolor='white')
ax2.set_xlabel('Avg km between halts (clipped at 40)')
ax2.set_ylabel('Number of trains')
ax2.set_title('How often does each train stop?')

plt.tight_layout()
plt.savefig('images/day1_trains.png', dpi=150)
plt.show()

In [ ]:
# Does a longer route mean stopping less often?
short = trains[trains['total_distance_km'] <  500]
long  = trains[trains['total_distance_km'] >= 1500]

print(f'Short trains (<500 km):   {len(short):,} trains | median stop interval: {short["avg_km_per_stop"].median():.1f} km')
print(f'Long trains  (>1500 km):  {len(long):,} trains | median stop interval: {long["avg_km_per_stop"].median():.1f} km')

**Exercise 5** — Find the 5 trains with the most stops. Print their train number and stop count.

In [ ]:
# Your code here


**Exercise 6** — How many trains cover more than 1,500 km AND stop every 15 km or less on average? These are long-distance trains that still stop very frequently.

In [ ]:
# Your code here


---
## Written questions

**Q1** — Look at the station map and the track map. Where are stations and connections densest? Name one region that looks sparse and suggest a reason — geography, population, or history.

*Your answer:*

**Q2** — The median gap between neighboring stations is about 7 km. What does this tell you about who Indian Railways was built to serve?

*Your answer:*

**Q3** — The busiest track connections are all very short and concentrated in one or two cities. Which cities? What kind of railway service runs on these connections?

*Your answer:*

**Q4** — A 2,000 km train stops almost as often as a 100 km train. Why might that be? What does it tell you about how the railway was designed?

*Your answer:*

**Q5** — `num_trains` counts train services, not passengers. Give one reason why a connection with 5 trains per day might carry more passengers than one with 50 trains per day. What does this tell you about the limits of the data?

*Your answer:*

---
**Next session:** we turn the track data into a network, compute degree and betweenness, and find the stations that actually control the flow of the entire country.